In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, base64, shutil, hashlib, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - write src/features_ugr16.py (pr DROPPED, hardened categorical
# encoder) and load the two weeks. Source = July week5, target = august_week1.
# =============================================================================
(config.PROJECT_ROOT/'src'/'features_ugr16.py').write_bytes(base64.b64decode("IiIiVUdSJzE2IGZlYXR1cmUgZW5jb2RpbmcuIFNpbmdsZSBzb3VyY2Ugb2YgdHJ1dGggZm9yIFVHUjE2IGZlYXR1cmUgbWF0cmljZXMuCk5ldEZsb3cgZmllbGRzOiB0ZSx0ZCxzYSxkYSxzcCxkcCxwcixmbGcsZndkLHN0b3MscGt0LGJ5dCxsYWJlbCAoK3dlZWspLgpwciAocHJvdG9jb2wpIGlzIERST1BQRUQ6IHN0cmluZyBpbiBqdWx5LCBkZXN0cm95ZWQgdG8gTmFOIGluIHRoZSBhdWd1c3QgcGFycXVldApieSBhbiBlYXJsaWVyIG51bWVyaWMgY29lcmNpb24sIHNvIG5vdCBjb21wYXJhYmxlIGFjcm9zcyB0aGUgcGFpciAoZGV2aWF0aW9uKS4KSWRlbnRpZmllcnMgKElQcykgYW5kIHRoZSB0aW1lc3RhbXAgYXJlIGV4Y2x1ZGVkLiBmbGcgaXMgY2F0ZWdvcmljYWwsIG9uZS1ob3QKYWdhaW5zdCBhIGZpeGVkIHZvY2FidWxhcnkgbGVhcm5lZCBvbiBzb3VyY2U7IHRoZSBlbmNvZGVyIG5vcm1hbGl6ZXMgbnVtZXJpYy0KbG9va2luZyBjYXRlZ29yeSB2YWx1ZXMgc28gaW50L2Zsb2F0IHN0b3JhZ2UgY2Fubm90IG1pc21hdGNoLiIiIgppbXBvcnQgbnVtcHkgYXMgbnAsIHBhbmRhcyBhcyBwZAoKRFJPUCAgICA9IFsndGUnLCdzYScsJ2RhJywncHInLCdsYWJlbCcsJ3dlZWsnLCdwYXJ0aXRpb24nXQpOVU1FUklDID0gWyd0ZCcsJ3NwJywnZHAnLCdmd2QnLCdzdG9zJywncGt0JywnYnl0J10KQ0FURUcgICA9IFsnZmxnJ10KCmRlZiBfY2F0KGRmLCBjKToKICAgIHMgPSBkZltjXQogICAgaWYgcGQuYXBpLnR5cGVzLmlzX251bWVyaWNfZHR5cGUocyk6CiAgICAgICAgcyA9IHMuYXN0eXBlKCdGbG9hdDY0JykuYXN0eXBlKCdzdHJpbmcnKS5zdHIucmVwbGFjZShyJ1wuMCQnLCAnJywgcmVnZXg9VHJ1ZSkKICAgIGVsc2U6CiAgICAgICAgcyA9IHMuYXN0eXBlKCdzdHJpbmcnKS5zdHIuc3RyaXAoKQogICAgcmV0dXJuIHMuZmlsbG5hKCduYScpLnJlcGxhY2Uoeyc8TkE+JzogJ25hJywgJ25hbic6ICduYScsICdOb25lJzogJ25hJywgJyc6ICduYSd9KS5hc3R5cGUoc3RyKQoKZGVmIG51bWVyaWNfZnJhbWUoZGYpOgogICAgcmV0dXJuIGRmW05VTUVSSUNdLmFwcGx5KHBkLnRvX251bWVyaWMsIGVycm9ycz0nY29lcmNlJykKCmRlZiBidWlsZF92b2NhYihkZiwgdG9wPTIwKToKICAgIHJldHVybiB7YzogbGlzdChfY2F0KGRmLCBjKS52YWx1ZV9jb3VudHMoKS5pbmRleFs6dG9wXSkgZm9yIGMgaW4gQ0FURUd9CgpkZWYgZW5jb2RlKGRmLCB2b2NhYik6CiAgICBYbiA9IG51bWVyaWNfZnJhbWUoZGYpLnRvX251bXB5KGR0eXBlPW5wLmZsb2F0NjQpCiAgICBYblt+bnAuaXNmaW5pdGUoWG4pXSA9IG5wLm5hbgogICAgY29scywgbmFtZXMgPSBbWG5dLCBsaXN0KE5VTUVSSUMpCiAgICBmb3IgYyBpbiBDQVRFRzoKICAgICAgICBzID0gX2NhdChkZiwgYykKICAgICAgICBmb3IgdiBpbiB2b2NhYltjXToKICAgICAgICAgICAgY29scy5hcHBlbmQoKHMgPT0gdikudG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NClbOiwgTm9uZV0pOyBuYW1lcy5hcHBlbmQoZid7Y309e3Z9JykKICAgIHJldHVybiBucC5oc3RhY2soY29scyksIG5hbWVzCg=="))
import importlib
if 'features_ugr16' in sys.modules: importlib.reload(sys.modules['features_ugr16'])
import features_ugr16 as fu

UGR = config.DATASETS_DIR / 'ugr16'
src = pd.read_parquet(UGR/'july_week5.parquet')       # SOURCE
tgt = pd.read_parquet(UGR/'august_week1.parquet')     # TARGET (drift rung)

for d in (src, tgt):
    d['label'] = d['label'].astype(str).str.strip().str.lower()
# blacklist not comparable (july stripped, august raw); anomaly-* detector-found.
SYNTH = ['dos','scan11','scan44','nerisbotnet']
KEEP  = ['background'] + SYNTH
src = src[src.label.isin(KEEP)].reset_index(drop=True)
tgt = tgt[tgt.label.isin(KEEP)].reset_index(drop=True)

print('SOURCE july_week5 :', len(src), src.label.value_counts().to_dict())
print('TARGET august_wk1 :', len(tgt), tgt.label.value_counts().to_dict())
print('features:', fu.NUMERIC + [f"flg(one-hot)"], '   (pr dropped)')


SOURCE july_week5 : 400000 {'background': 200000, 'dos': 50000, 'scan11': 50000, 'scan44': 50000, 'nerisbotnet': 50000}
TARGET august_wk1 : 400000 {'background': 200000, 'dos': 50000, 'scan11': 50000, 'scan44': 50000, 'nerisbotnet': 50000}
features: ['td', 'sp', 'dp', 'fwd', 'stos', 'pkt', 'byt', 'flg(one-hot)']    (pr dropped)


In [3]:
# =============================================================================
# Cell 3 - five-partition split of the SOURCE (July), stratified by class
# (preregistration 4). D_eval and T_cal come from the TARGET (August) later.
# =============================================================================
PARTITION_SEED_UGR = 20260725
def stratified_split(df, fractions, seed, col='label'):
    rng=np.random.default_rng(seed); names=list(fractions)
    fr=np.array([fractions[k] for k in names],float); big=names[int(np.argmax(fr))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(fr*n).astype(int); c[names.index(big)]+=n-c.sum(); k=0
        for nm,q in zip(names,c): a.loc[idx[k:k+q]]=nm; k+=q
    return a

src = src.assign(partition=stratified_split(src, config.SPLIT_FRACTIONS, PARTITION_SEED_UGR).values)
parts = {k: src[src.partition==k] for k in config.SPLIT_FRACTIONS}
assert sum(len(p) for p in parts.values()) == len(src)
S_pool = parts['source_cal_pool']
for k,p in parts.items(): print(f'{k:16s} {len(p):8d}')


train              240000
val                 40000
probcal             60000
source_cal_pool     60000


In [4]:
# =============================================================================
# Cell 4 - BINDING feasibility + focal class. 50k caps flattened natural rarity,
# so focal is chosen by TRUE synthetic-attack rarity (nerisbotnet), recorded with
# rationale, not by capped count. Feasible = source_cal_pool count >= ceil(1/a)-1.
# =============================================================================
alphas = [config.ALPHA_PRIMARY] + config.ALPHA_SENSITIVITY + config.ALPHA_CONDITIONAL
scal = S_pool['label'].value_counts(); need = config.min_calib_n(config.ALPHA_PRIMARY)
rows=[]
for a in alphas:
    nd = config.min_calib_n(a)
    for cls in scal.index:
        rows.append({'dataset':'ugr16','environment':'july_to_august','alpha':a,'class':cls,
                     'source_calib_n':int(scal[cls]),'min_calib_needed':nd,'feasible':int(scal[cls])>=nd})
feas=pd.DataFrame(rows); feas.to_csv(config.REPORTS_DIR/'feasibility_binding_ugr16.csv',index=False)
print(feas[feas.alpha==config.ALPHA_PRIMARY].to_string(index=False))

FOCAL_CLASS = 'nerisbotnet'
sel = feas[(feas.alpha==config.ALPHA_PRIMARY)&(feas['class']==FOCAL_CLASS)]
assert len(sel)==1 and bool(sel['feasible'].iloc[0]), 'focal class missing or not feasible'
record = {'dataset':'ugr16','environment':'july_to_august','focal_class':FOCAL_CLASS,
          'alpha_primary':config.ALPHA_PRIMARY,
          'selection_rule':('capped counts (50k each) flattened natural rarity, so focal is chosen by '
                            'true operational rarity + temporal structure of the UGR-16 synthetic attacks '
                            '(nerisbotnet: rarest, bursty single botnet), not by capped count; before any coverage'),
          'source_calib_counts':{c:int(scal[c]) for c in scal.index},
          'min_calib_needed':need,'partition_seed':PARTITION_SEED_UGR,
          'status':'BINDING - no UGR16 coverage number computed at time of writing'}
(config.REPORTS_DIR/'focal_class_record_ugr16.json').write_text(json.dumps(record,indent=2))
print('\n', json.dumps(record, indent=2))


dataset    environment  alpha       class  source_calib_n  min_calib_needed  feasible
  ugr16 july_to_august   0.05  background           30000                19      True
  ugr16 july_to_august   0.05         dos            7500                19      True
  ugr16 july_to_august   0.05      scan11            7500                19      True
  ugr16 july_to_august   0.05      scan44            7500                19      True
  ugr16 july_to_august   0.05 nerisbotnet            7500                19      True

 {
  "dataset": "ugr16",
  "environment": "july_to_august",
  "focal_class": "nerisbotnet",
  "alpha_primary": 0.05,
  "selection_rule": "capped counts (50k each) flattened natural rarity, so focal is chosen by true operational rarity + temporal structure of the UGR-16 synthetic attacks (nerisbotnet: rarest, bursty single botnet), not by capped count; before any coverage",
  "source_calib_counts": {
    "background": 30000,
    "dos": 7500,
    "scan11": 7500,
    "scan44": 7500

In [ ]:
# =============================================================================
# Cell 5 - July->August shift measurement (preregistration 9), on the 9 clean
# flow features (pr dropped). S_cov: cross-fitted domain classifier, source-calib
# vs target covariates. S_lab: TV distance of class priors. S_sup: target mass in
# zero-source-support classes (~0, shared attacks). Plus the permutation null.
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

vocab = fu.build_vocab(parts['train'])
Xsp,names = fu.encode(S_pool, vocab); Xtg,_ = fu.encode(tgt, vocab)
SUB = config.SCOV_SUBSAMPLE_PER_SIDE
print('feature dim:', Xsp.shape[1], '| any all-zero column source:',
      int((Xsp==0).all(0).sum()), 'target:', int((Xtg==0).all(0).sum()))

def s_cov_cv(Xc, Xe, seed, n=None, folds=5):
    n = SUB if n is None else n; rng=np.random.default_rng(seed)
    def sub(X): return X[rng.choice(len(X),n,replace=False)] if len(X)>n else X
    Xc2,Xe2=sub(Xc),sub(Xe); X=np.vstack([Xc2,Xe2]); y=np.r_[np.zeros(len(Xc2)),np.ones(len(Xe2))]
    skf=StratifiedKFold(n_splits=folds,shuffle=True,random_state=seed); a=[]
    for tr,te in skf.split(X,y):
        clf=HistGradientBoostingClassifier(max_depth=4,max_iter=150,random_state=seed)
        clf.fit(X[tr],y[tr]); a.append(roc_auc_score(y[te],clf.predict_proba(X[te])[:,1]))
    return float(np.mean(a))

def s_cov_single(Xa,Xb,seed,n):
    rng=np.random.default_rng(seed)
    def sub(X): return X[rng.choice(len(X),n,replace=False)] if len(X)>n else X
    Xa2,Xb2=sub(Xa),sub(Xb); X=np.vstack([Xa2,Xb2]); y=np.r_[np.zeros(len(Xa2)),np.ones(len(Xb2))]
    idx=rng.permutation(len(X)); cut=len(X)//2
    clf=HistGradientBoostingClassifier(max_depth=4,max_iter=150,random_state=seed)
    clf.fit(X[idx[:cut]],y[idx[:cut]]); return float(roc_auc_score(y[idx[cut:]],clf.predict_proba(X[idx[cut:]])[:,1]))

NULL_SUB=8000; NULL_DRAWS=config.PERMUTATION_NULL_DRAWS; Xpool=Xsp; vals=[]
for i in range(NULL_DRAWS):
    rng=np.random.default_rng(30000+i); idx=rng.permutation(len(Xpool)); h=len(idx)//2
    vals.append(s_cov_single(Xpool[idx[:h]],Xpool[idx[h:]],40000+i,NULL_SUB))
    if (i+1)%50==0: print(f'  null {i+1}/{NULL_DRAWS}')
NULL_Q95=float(np.quantile(vals,config.PERMUTATION_NULL_Q))

def priors(df, classes):
    v=df['label'].value_counts(normalize=True); return {c:float(v.get(c,0.0)) for c in classes}
CLASSES=sorted(src['label'].unique())
scov=s_cov_cv(Xsp,Xtg,seed=515)
slab=0.5*sum(abs(priors(S_pool,CLASSES)[c]-priors(tgt,CLASSES)[c]) for c in CLASSES)
ssup=float((~tgt['label'].isin(S_pool['label'].unique())).mean())
shift={'dataset':'ugr16','environment':'july_to_august','n_features':int(Xsp.shape[1]),
       'S_cov':round(scov,4),'S_cov_above_null':bool(scov>NULL_Q95),'null_q95':round(NULL_Q95,4),
       'S_lab':round(slab,4),'S_sup':round(ssup,4),
       'note':'fixed-support temporal covariate shift on 9 flow features; pr dropped; shared 4 attacks so S_sup~0'}
(config.REPORTS_DIR/'ladder_shift_measures_ugr16.json').write_text(json.dumps(shift,indent=2))
print('\n', json.dumps(shift, indent=2))


feature dim: 27 | any all-zero column source: 1 target: 1
  null 50/200
  null 100/200
  null 150/200
  null 200/200


In [ ]:
# =============================================================================
# Cell 6 - fingerprints, provenance note, commit.
# =============================================================================
config.PROC_DIR.mkdir(parents=True, exist_ok=True)
def fp(idx): return hashlib.sha256(np.sort(np.asarray(idx,np.int64)).tobytes()).hexdigest()
fps={}
for k,p in parts.items():
    np.save(config.PROC_DIR/f'ugr16_src_{k}_idx.npy', np.sort(p.index.to_numpy()))
    fps[f'source/{k}']={'n':int(len(p)),'sha256':fp(p.index)}
np.save(config.PROC_DIR/'ugr16_target_idx.npy', np.arange(len(tgt)))
(config.REPORTS_DIR/'partition_fingerprints_ugr16.json').write_text(json.dumps(
    {'partition_seed':PARTITION_SEED_UGR,'environment':'july_to_august',
     'source':'july_week5','target':'august_week1','fingerprints':fps},indent=2))

dev=config.REPORTS_DIR/'deviations.md'
note=('\n## nb15 (UGR16) - source july_week5 is the uniqblacklistremoved variant, target august_week1 '
      'is the raw full file; blacklist dropped (not comparable). pr (protocol) dropped from features: '
      'string in july but destroyed to NaN in the august parquet by an earlier numeric coercion, so not '
      'comparable; S_cov computed on 9 flow features. Focal = nerisbotnet by true rarity, not the 50k caps. '
      'All before any coverage.\n')
if dev.exists() and 'nb15 (UGR16)' not in dev.read_text():
    with open(dev,'a') as f: f.write(note)

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb15: UGR16 July->August partitions, feasibility, focal (nerisbotnet), shift; pr dropped')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
